# LangGraph Custom State Machines
## Document Processing Pipeline

Previous notebooks focused on multi-agent routing patterns. This notebook takes a different approach: modeling a **document processing pipeline** as an explicit state machine with defined lifecycle stages.

### State Machines vs Agent Routing

| Feature | Agent Routing (Supervisor) | State Machine |
|---------|--------------------------|---------------|
| Transitions | LLM decides next step | **Explicit, code-defined rules** |
| Lifecycle | Implicit | **Explicit stages (enum)** |
| Error handling | Ad-hoc | **Structured retry + recovery** |
| Audit trail | Messages only | **Processing log with timestamps** |
| Branching | Binary (this agent or that) | **Multi-way (3+ paths)** |
| Predictability | Variable | **Highly predictable** |

### What We're Building

A document processing system where:
- Documents enter through an **intake** node
- A **classifier** identifies the document type (invoice, contract, support ticket)
- Type-specific **extractors** pull structured data
- A **validation gate** checks extraction quality with three-way routing:
  - Pass → Approval
  - Fail + retries remaining → Error handler → Retry extraction
  - Fail + no retries → Rejection
- Every step is logged in an **audit trail**

### Architecture
```
intake → classifier →┬→ invoice_extractor ──→┐
                      ├→ contract_analyzer ──→├→ validation_gate →┬→ approval → END
                      └→ ticket_parser ──────→┘                   ├→ error_handler → (retry)
                                                                   └→ rejection → END
```

In [ ]:
%pip install -q langgraph langchain langchain-openai langchain-community

In [ ]:
import os
import getpass
import operator
from typing import Annotated, TypedDict, Literal, Any
from enum import Enum
from datetime import datetime

from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup complete!")

## 1. The State Machine Mental Model

In a state machine, every entity has an **explicit lifecycle stage**. A document progresses through well-defined stages, and the rules for transitioning between stages are explicit in code, not decided by an LLM.

### Document Lifecycle

```
INTAKE → CLASSIFYING → PROCESSING → VALIDATING → APPROVED
                                         │            or
                                         └── RETRY → PROCESSING (back to extraction)
                                         │
                                         └── REJECTED
```

### Why Enums for Stages?

Using an `Enum` for stages gives us:
1. **Type safety** — Can't set an invalid stage
2. **Self-documenting** — The enum IS the documentation of valid stages
3. **Conditional routing** — Easy to branch on stage values
4. **Audit clarity** — Logs show exact stage transitions

In [ ]:
class DocStage(str, Enum):
    """Explicit lifecycle stages for document processing."""
    INTAKE = "intake"
    CLASSIFYING = "classifying"
    PROCESSING = "processing"
    VALIDATING = "validating"
    RETRY = "retry"
    APPROVED = "approved"
    REJECTED = "rejected"
    ERROR = "error"

def append_log(existing: list, new: list) -> list:
    """Reducer that appends new log entries."""
    return existing + new

class DocumentState(TypedDict):
    messages: Annotated[list, add_messages]
    stage: str  # DocStage value
    document_type: str  # "invoice", "contract", "ticket"
    document_content: str  # Raw document text
    extracted_data: dict  # Structured data pulled from document
    validation_errors: list  # List of validation error strings
    retry_count: int
    max_retries: int
    processing_log: Annotated[list, append_log]  # Audit trail

print("DocStage enum values:", [s.value for s in DocStage])
print(f"\nDocumentState fields: {list(DocumentState.__annotations__.keys())}")

## 2. Audit Trail Helper

Every node logs its actions to the `processing_log`. This creates a complete audit trail of what happened to each document and when.

In [ ]:
def log_entry(stage: str, action: str, details: str = "") -> dict:
    """Create a timestamped log entry."""
    return {
        "timestamp": datetime.now().isoformat(),
        "stage": stage,
        "action": action,
        "details": details,
    }

# Example
example_log = log_entry(DocStage.INTAKE.value, "Document received", "Invoice from Acme Corp")
print("Example log entry:")
for key, value in example_log.items():
    print(f"  {key}: {value}")

## 3. Intake & Classification Nodes

The intake node receives the raw document and prepares it for processing. The classifier uses the LLM to determine what type of document it is, which determines the processing path.

In [ ]:
def intake_node(state: DocumentState) -> dict:
    """Receive and log the incoming document."""
    content = state.get("document_content", "")
    return {
        "stage": DocStage.CLASSIFYING.value,
        "processing_log": [log_entry(
            DocStage.INTAKE.value, 
            "Document received",
            f"Length: {len(content)} chars"
        )],
        "retry_count": 0,
        "max_retries": 2,
        "validation_errors": [],
        "extracted_data": {},
    }

print("Intake node defined")

In [ ]:
class DocumentClassification(BaseModel):
    """Classification of a document."""
    document_type: Literal["invoice", "contract", "ticket"] = Field(
        description="The type of document"
    )
    confidence: float = Field(description="Classification confidence 0-1", ge=0.0, le=1.0)
    reasoning: str = Field(description="Why this classification was chosen")

classifier_llm = llm.with_structured_output(DocumentClassification)

def classifier_node(state: DocumentState) -> dict:
    """Classify the document type using LLM."""
    classification = classifier_llm.invoke([
        SystemMessage(content="""Classify this document as one of:
- invoice: Bills, payment requests, purchase orders, receipts
- contract: Legal agreements, terms of service, NDAs, employment contracts
- ticket: Support tickets, bug reports, feature requests, complaints

Analyze the content and structure to determine the type."""),
        HumanMessage(content=f"Document to classify:\n\n{state['document_content']}"),
    ])
    
    return {
        "document_type": classification.document_type,
        "stage": DocStage.PROCESSING.value,
        "processing_log": [log_entry(
            DocStage.CLASSIFYING.value,
            f"Classified as: {classification.document_type}",
            f"Confidence: {classification.confidence:.0%}. {classification.reasoning}"
        )],
        "messages": [AIMessage(content=f"[Classifier] Document type: {classification.document_type} (confidence: {classification.confidence:.0%})")],
    }

print("Classifier node defined with structured output")

In [ ]:
def route_by_document_type(state: DocumentState) -> str:
    """Three-way routing based on document type."""
    doc_type = state.get("document_type", "")
    routes = {
        "invoice": "invoice_extractor",
        "contract": "contract_analyzer",
        "ticket": "ticket_parser",
    }
    return routes.get(doc_type, "ticket_parser")  # Default fallback

print("Three-way routing function defined: invoice → invoice_extractor, contract → contract_analyzer, ticket → ticket_parser")

## 4. Type-Specific Processing Nodes

Each document type has its own extraction logic. The LLM extracts structured data appropriate to the document type.

- **Invoice Extractor** → Pulls vendor, amount, date, line items
- **Contract Analyzer** → Extracts parties, terms, key clauses
- **Ticket Parser** → Identifies issue type, priority, affected system

In [ ]:
class InvoiceData(BaseModel):
    """Structured data extracted from an invoice."""
    vendor_name: str = Field(description="Name of the vendor/supplier")
    invoice_number: str = Field(description="Invoice or reference number")
    total_amount: str = Field(description="Total amount due")
    currency: str = Field(description="Currency code (e.g., USD, EUR)")
    due_date: str = Field(description="Payment due date")
    line_items: list[str] = Field(description="List of line item descriptions")

invoice_llm = llm.with_structured_output(InvoiceData)

def invoice_extractor(state: DocumentState) -> dict:
    """Extract structured data from an invoice."""
    try:
        data = invoice_llm.invoke([
            SystemMessage(content="Extract all invoice fields from this document. If a field is not present, use 'N/A'."),
            HumanMessage(content=state["document_content"]),
        ])
        
        return {
            "extracted_data": data.model_dump(),
            "stage": DocStage.VALIDATING.value,
            "processing_log": [log_entry(
                DocStage.PROCESSING.value,
                "Invoice data extracted",
                f"Vendor: {data.vendor_name}, Amount: {data.total_amount}"
            )],
            "messages": [AIMessage(content=f"[Invoice Extractor] Extracted: {data.vendor_name}, {data.total_amount} {data.currency}")],
        }
    except Exception as e:
        return {
            "stage": DocStage.ERROR.value,
            "processing_log": [log_entry(DocStage.PROCESSING.value, "Extraction failed", str(e))],
            "validation_errors": [f"Invoice extraction error: {str(e)}"],
        }

print("Invoice extractor defined — extracts vendor, amount, date, line items")

In [ ]:
class ContractData(BaseModel):
    """Structured data extracted from a contract."""
    parties: list[str] = Field(description="Parties involved in the contract")
    contract_type: str = Field(description="Type of contract (NDA, service agreement, etc.)")
    effective_date: str = Field(description="When the contract takes effect")
    termination_date: str = Field(description="When the contract expires")
    key_terms: list[str] = Field(description="Key terms and obligations")
    risk_flags: list[str] = Field(description="Any concerning clauses or risks identified")

contract_llm = llm.with_structured_output(ContractData)

def contract_analyzer(state: DocumentState) -> dict:
    """Extract structured data from a contract."""
    try:
        data = contract_llm.invoke([
            SystemMessage(content="""Extract contract details. Pay special attention to:
- All parties and their obligations
- Important dates and deadlines
- Any unusual or risky clauses (indemnification, non-compete, auto-renewal, liability caps)
If a field is not present, use 'N/A'."""),
            HumanMessage(content=state["document_content"]),
        ])
        
        return {
            "extracted_data": data.model_dump(),
            "stage": DocStage.VALIDATING.value,
            "processing_log": [log_entry(
                DocStage.PROCESSING.value,
                "Contract data extracted",
                f"Type: {data.contract_type}, Parties: {', '.join(data.parties)}"
            )],
            "messages": [AIMessage(content=f"[Contract Analyzer] Type: {data.contract_type}, Parties: {', '.join(data.parties)}, Risk flags: {len(data.risk_flags)}")],
        }
    except Exception as e:
        return {
            "stage": DocStage.ERROR.value,
            "processing_log": [log_entry(DocStage.PROCESSING.value, "Extraction failed", str(e))],
            "validation_errors": [f"Contract extraction error: {str(e)}"],
        }

print("Contract analyzer defined — extracts parties, terms, risk flags")

In [ ]:
class TicketData(BaseModel):
    """Structured data extracted from a support ticket."""
    issue_type: Literal["bug", "feature_request", "complaint", "question", "other"] = Field(
        description="Type of support issue"
    )
    priority: Literal["critical", "high", "medium", "low"] = Field(
        description="Priority level based on impact and urgency"
    )
    affected_system: str = Field(description="System or component affected")
    summary: str = Field(description="One-sentence summary of the issue")
    reproduction_steps: list[str] = Field(description="Steps to reproduce (for bugs) or details")

ticket_llm = llm.with_structured_output(TicketData)

def ticket_parser(state: DocumentState) -> dict:
    """Extract structured data from a support ticket."""
    try:
        data = ticket_llm.invoke([
            SystemMessage(content="""Parse this support ticket. Determine:
- Issue type (bug, feature_request, complaint, question, other)
- Priority based on severity and business impact
- Affected system/component
- Clear summary
- Steps to reproduce or key details"""),
            HumanMessage(content=state["document_content"]),
        ])
        
        return {
            "extracted_data": data.model_dump(),
            "stage": DocStage.VALIDATING.value,
            "processing_log": [log_entry(
                DocStage.PROCESSING.value,
                "Ticket data extracted",
                f"Type: {data.issue_type}, Priority: {data.priority}"
            )],
            "messages": [AIMessage(content=f"[Ticket Parser] Type: {data.issue_type}, Priority: {data.priority}, System: {data.affected_system}")],
        }
    except Exception as e:
        return {
            "stage": DocStage.ERROR.value,
            "processing_log": [log_entry(DocStage.PROCESSING.value, "Extraction failed", str(e))],
            "validation_errors": [f"Ticket extraction error: {str(e)}"],
        }

print("Ticket parser defined — extracts issue type, priority, affected system")

## 5. The Validation Gate

This is the most interesting node in our state machine. It performs **three-way routing**:

1. **PASS** → Data is valid → Route to approval
2. **FAIL + retries remaining** → Route to error handler (which retries extraction)
3. **FAIL + max retries reached** → Route to rejection

This pattern is extremely common in production systems but rarely shown in tutorials.

In [ ]:
class ValidationResult(BaseModel):
    """Result of validating extracted data."""
    is_valid: bool = Field(description="Whether the extracted data passes validation")
    errors: list[str] = Field(description="List of validation errors found")
    completeness_score: float = Field(description="How complete the extraction is, 0.0-1.0", ge=0.0, le=1.0)

validation_llm = llm.with_structured_output(ValidationResult)

def validation_gate(state: DocumentState) -> dict:
    """Validate extracted data quality with three-way routing."""
    extracted = state.get("extracted_data", {})
    doc_type = state.get("document_type", "unknown")
    
    validation = validation_llm.invoke([
        SystemMessage(content=f"""Validate the extracted {doc_type} data for quality and completeness.

Check for:
1. Missing required fields (values of 'N/A' or empty)
2. Implausible values (negative amounts, dates in wrong format)
3. Inconsistencies between fields
4. Overall completeness — did we capture the essential information?

A completeness score above 0.7 is acceptable. Below 0.7 needs retry."""),
        HumanMessage(content=f"Extracted data:\n{extracted}"),
    ])
    
    if validation.is_valid and validation.completeness_score >= 0.7:
        return {
            "stage": DocStage.APPROVED.value,
            "validation_errors": [],
            "processing_log": [log_entry(
                DocStage.VALIDATING.value,
                "Validation PASSED",
                f"Completeness: {validation.completeness_score:.0%}"
            )],
            "messages": [AIMessage(content=f"[Validation] PASSED (completeness: {validation.completeness_score:.0%})")],
        }
    else:
        retry_count = state.get("retry_count", 0)
        max_retries = state.get("max_retries", 2)
        
        if retry_count < max_retries:
            return {
                "stage": DocStage.RETRY.value,
                "validation_errors": validation.errors,
                "processing_log": [log_entry(
                    DocStage.VALIDATING.value,
                    f"Validation FAILED — retry {retry_count + 1}/{max_retries}",
                    f"Errors: {', '.join(validation.errors)}"
                )],
                "messages": [AIMessage(content=f"[Validation] FAILED — retrying ({retry_count + 1}/{max_retries}). Errors: {', '.join(validation.errors)}")],
            }
        else:
            return {
                "stage": DocStage.REJECTED.value,
                "validation_errors": validation.errors,
                "processing_log": [log_entry(
                    DocStage.VALIDATING.value,
                    "Validation FAILED — max retries exceeded",
                    f"Errors: {', '.join(validation.errors)}"
                )],
                "messages": [AIMessage(content=f"[Validation] REJECTED after {max_retries} retries. Errors: {', '.join(validation.errors)}")],
            }

print("Validation gate defined with three-way routing: pass / retry / reject")

In [ ]:
def error_handler(state: DocumentState) -> dict:
    """Handle validation failures by preparing for retry."""
    retry_count = state.get("retry_count", 0)
    errors = state.get("validation_errors", [])
    
    return {
        "retry_count": retry_count + 1,
        "stage": DocStage.PROCESSING.value,
        "processing_log": [log_entry(
            DocStage.RETRY.value,
            f"Retry #{retry_count + 1} initiated",
            f"Previous errors to fix: {', '.join(errors)}"
        )],
        "messages": [AIMessage(content=f"[Error Handler] Initiating retry #{retry_count + 1}. Will address: {', '.join(errors)}")],
    }

def approval_node(state: DocumentState) -> dict:
    """Mark document as approved."""
    doc_type = state.get("document_type", "unknown")
    extracted = state.get("extracted_data", {})
    
    response = llm.invoke([
        SystemMessage(content=f"Write a brief approval summary for this {doc_type}. Include the key extracted data points."),
        HumanMessage(content=f"Extracted data: {extracted}"),
    ])
    
    return {
        "stage": DocStage.APPROVED.value,
        "processing_log": [log_entry(
            DocStage.APPROVED.value,
            "Document APPROVED",
            response.content[:200]
        )],
        "messages": [AIMessage(content=f"[Approved] {response.content}")],
    }

def rejection_node(state: DocumentState) -> dict:
    """Mark document as rejected with reasons."""
    errors = state.get("validation_errors", [])
    
    return {
        "stage": DocStage.REJECTED.value,
        "processing_log": [log_entry(
            DocStage.REJECTED.value,
            "Document REJECTED",
            f"Final errors: {', '.join(errors)}"
        )],
        "messages": [AIMessage(content=f"[Rejected] Document could not be processed. Errors: {', '.join(errors)}")],
    }

print("Error handler, approval, and rejection nodes defined")

## 6. Full Graph Assembly

The routing logic is the most important part:

1. **After classifier**: Route to type-specific extractor (3-way)
2. **After validation**: Route based on stage — approved, retry, or rejected (3-way)
3. **After error handler**: Route back to the correct extractor based on document type (3-way)

In [ ]:
def route_after_validation(state: DocumentState) -> str:
    """Three-way routing after validation."""
    stage = state.get("stage", "")
    if stage == DocStage.APPROVED.value:
        return "approval"
    elif stage == DocStage.RETRY.value:
        return "error_handler"
    else:  # REJECTED
        return "rejection"

def route_after_error(state: DocumentState) -> str:
    """Route back to the correct extractor for retry."""
    doc_type = state.get("document_type", "")
    routes = {
        "invoice": "invoice_extractor",
        "contract": "contract_analyzer",
        "ticket": "ticket_parser",
    }
    return routes.get(doc_type, "ticket_parser")

# Build the graph
graph = StateGraph(DocumentState)

# Add all nodes
graph.add_node("intake", intake_node)
graph.add_node("classifier", classifier_node)
graph.add_node("invoice_extractor", invoice_extractor)
graph.add_node("contract_analyzer", contract_analyzer)
graph.add_node("ticket_parser", ticket_parser)
graph.add_node("validation_gate", validation_gate)
graph.add_node("error_handler", error_handler)
graph.add_node("approval", approval_node)
graph.add_node("rejection", rejection_node)

# Wire the graph
graph.add_edge(START, "intake")
graph.add_edge("intake", "classifier")

# Classifier → type-specific extractor
graph.add_conditional_edges("classifier", route_by_document_type, {
    "invoice_extractor": "invoice_extractor",
    "contract_analyzer": "contract_analyzer",
    "ticket_parser": "ticket_parser",
})

# All extractors → validation
graph.add_edge("invoice_extractor", "validation_gate")
graph.add_edge("contract_analyzer", "validation_gate")
graph.add_edge("ticket_parser", "validation_gate")

# Validation → approval / error_handler / rejection
graph.add_conditional_edges("validation_gate", route_after_validation, {
    "approval": "approval",
    "error_handler": "error_handler",
    "rejection": "rejection",
})

# Error handler → back to correct extractor
graph.add_conditional_edges("error_handler", route_after_error, {
    "invoice_extractor": "invoice_extractor",
    "contract_analyzer": "contract_analyzer",
    "ticket_parser": "ticket_parser",
})

# Terminal nodes
graph.add_edge("approval", END)
graph.add_edge("rejection", END)

app = graph.compile()
print("Document processing pipeline compiled!")

In [ ]:
from IPython.display import display, Image

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())

## 7. Scenario 1: Clean Invoice Processing

A well-formatted invoice should flow cleanly through: intake → classify → extract → validate → approve.

In [ ]:
invoice_doc = """
INVOICE #INV-2024-0847

From: CloudTech Solutions Inc.
      123 Silicon Valley Blvd
      San Jose, CA 95134

To: Acme Corporation
    456 Business Park Dr
    Austin, TX 78701

Date: January 15, 2024
Due Date: February 15, 2024

Description                          Qty    Unit Price    Total
─────────────────────────────────────────────────────────────
Enterprise Cloud Platform License     1     $12,000.00    $12,000.00
Premium Support Package (Annual)      1      $3,600.00     $3,600.00
Data Migration Service               1      $2,500.00     $2,500.00
Custom Integration Development       40hrs     $150.00     $6,000.00
─────────────────────────────────────────────────────────────
                                              Subtotal:   $24,100.00
                                              Tax (8.25%): $1,988.25
                                              TOTAL:      $26,088.25

Payment Terms: Net 30
Payment Method: Wire transfer to CloudTech Solutions, Account ending 4821
"""

result1 = app.invoke({
    "messages": [],
    "stage": "",
    "document_type": "",
    "document_content": invoice_doc,
    "extracted_data": {},
    "validation_errors": [],
    "retry_count": 0,
    "max_retries": 2,
    "processing_log": [],
})

print("=" * 80)
print("SCENARIO 1: Clean Invoice Processing")
print("=" * 80)

for msg in result1["messages"]:
    if isinstance(msg, AIMessage):
        print(f"\n  {msg.content[:300]}")

print(f"\n  Final Stage: {result1.get('stage')}")
print(f"  Document Type: {result1.get('document_type')}")
print(f"\n  Extracted Data:")
for key, value in result1.get("extracted_data", {}).items():
    print(f"    {key}: {value}")

## 8. Scenario 2: Contract with Risk Flags

A contract with potentially risky clauses. The system should identify and flag them.

In [ ]:
contract_doc = """
MASTER SERVICE AGREEMENT

This Master Service Agreement ("Agreement") is entered into as of March 1, 2024.

BETWEEN:
  DataFlow Analytics Inc. ("Provider"), a Delaware corporation
  AND
  MegaCorp International Ltd. ("Client"), a UK limited company

1. SERVICES: Provider shall deliver AI-powered data analytics platform access, 
   including real-time dashboards, predictive modeling, and API access.

2. TERM: This Agreement is effective for 36 months from the Effective Date, 
   with automatic renewal for successive 12-month periods unless terminated 
   with 90 days written notice.

3. FEES: Client shall pay $45,000 per month. Fees increase by 8% annually.
   Late payments incur 2% monthly interest.

4. INDEMNIFICATION: Client shall indemnify Provider against ALL claims, losses, 
   and damages arising from Client's use of the Services, without limitation.
   Provider's total liability shall not exceed fees paid in the prior 3 months.

5. NON-COMPETE: Client agrees not to develop or use competing analytics 
   solutions for a period of 24 months after termination.

6. DATA RIGHTS: All data processed through the platform, including derived 
   insights and models, shall be jointly owned by Provider and Client.

7. TERMINATION: Provider may terminate for any reason with 30 days notice. 
   Client may terminate only for material breach with 90 days cure period.

8. GOVERNING LAW: This Agreement shall be governed by the laws of Delaware.

SIGNATURES:
DataFlow Analytics Inc. - [Signature]     Date: March 1, 2024
MegaCorp International Ltd. - [Signature]  Date: March 1, 2024
"""

result2 = app.invoke({
    "messages": [],
    "stage": "",
    "document_type": "",
    "document_content": contract_doc,
    "extracted_data": {},
    "validation_errors": [],
    "retry_count": 0,
    "max_retries": 2,
    "processing_log": [],
})

print("=" * 80)
print("SCENARIO 2: Contract with Risk Flags")
print("=" * 80)

for msg in result2["messages"]:
    if isinstance(msg, AIMessage):
        print(f"\n  {msg.content[:300]}")

print(f"\n  Final Stage: {result2.get('stage')}")
print(f"\n  Extracted Data:")
for key, value in result2.get("extracted_data", {}).items():
    if isinstance(value, list):
        print(f"    {key}:")
        for item in value:
            print(f"      - {item[:100]}")
    else:
        print(f"    {key}: {value}")

## 9. Scenario 3: High-Priority Support Ticket

A critical support ticket that needs proper priority scoring and routing.

In [ ]:
ticket_doc = """
SUPPORT TICKET #TKT-2024-5291

Submitted by: Sarah Chen, VP of Engineering (sarah.chen@megacorp.com)
Company: MegaCorp International (Enterprise Plan)
Date: 2024-01-20 03:47 AM UTC

Subject: CRITICAL - Production API returning 500 errors, all services down

Description:
Starting at approximately 2:30 AM UTC, our production environment began 
receiving intermittent 500 errors from the Analytics API (endpoint: 
/api/v2/analytics/realtime). By 3:00 AM, the error rate had increased 
to 100% — all API calls are failing.

Impact:
- Our real-time dashboard is completely down for all 10,000+ users
- Three downstream services that depend on the Analytics API are failing
- Our SLA requires 99.99% uptime, we are currently in violation
- Estimated revenue impact: $50,000/hour

Steps to Reproduce:
1. Make any GET request to /api/v2/analytics/realtime
2. Include valid Bearer token in Authorization header
3. Response: HTTP 500 with body {"error": "internal_server_error", "trace_id": "abc-123-def"}

What we've tried:
- Verified our API keys are valid and not expired
- Checked our network connectivity (other APIs work fine)
- Confirmed the issue affects all our API keys, not just one
- Checked status.example.com — shows all green (seems inaccurate)

This is a P0 issue requiring immediate escalation.
"""

result3 = app.invoke({
    "messages": [],
    "stage": "",
    "document_type": "",
    "document_content": ticket_doc,
    "extracted_data": {},
    "validation_errors": [],
    "retry_count": 0,
    "max_retries": 2,
    "processing_log": [],
})

print("=" * 80)
print("SCENARIO 3: Critical Support Ticket")
print("=" * 80)

for msg in result3["messages"]:
    if isinstance(msg, AIMessage):
        print(f"\n  {msg.content[:300]}")

print(f"\n  Final Stage: {result3.get('stage')}")
print(f"\n  Extracted Data:")
for key, value in result3.get("extracted_data", {}).items():
    if isinstance(value, list):
        print(f"    {key}:")
        for item in value:
            print(f"      - {item[:100]}")
    else:
        print(f"    {key}: {value}")

## 10. Processing Log / Audit Trail

One of the most valuable features of the state machine approach is the automatic audit trail. Let's inspect the processing logs from all three scenarios.

In [ ]:
print("=" * 80)
print("PROCESSING AUDIT TRAILS")
print("=" * 80)

for name, result_data in [("Invoice", result1), ("Contract", result2), ("Ticket", result3)]:
    print(f"\n{'─' * 60}")
    print(f"  {name} — Final Stage: {result_data.get('stage', 'unknown')}")
    print(f"{'─' * 60}")
    
    logs = result_data.get("processing_log", [])
    for i, entry in enumerate(logs):
        timestamp = entry.get("timestamp", "")
        # Just show time portion for readability
        time_str = timestamp.split("T")[1][:8] if "T" in timestamp else timestamp
        stage = entry.get("stage", "?")
        action = entry.get("action", "?")
        details = entry.get("details", "")
        
        print(f"  [{time_str}] {stage:>12} | {action}")
        if details:
            print(f"  {'':>12}   {'':>12} | {details[:80]}")
    
    retries = result_data.get("retry_count", 0)
    if retries > 0:
        print(f"\n  Retries used: {retries}")

## 11. Stage Transition Visualization

Let's visualize the exact path each document took through the state machine.

In [ ]:
print("STAGE TRANSITIONS")
print("=" * 60)

for name, result_data in [("Invoice", result1), ("Contract", result2), ("Ticket", result3)]:
    logs = result_data.get("processing_log", [])
    stages = [entry.get("stage", "?") for entry in logs]
    
    print(f"\n  {name}:")
    print(f"  {' → '.join(stages)}")
    print(f"  ({len(stages)} steps, {result_data.get('retry_count', 0)} retries)")

## 12. Key Takeaways

### What We Built

A document processing pipeline with:
- **Explicit lifecycle stages** via `DocStage` enum
- **Type-specific processing** with three extraction paths
- **Three-way validation routing** (approve / retry / reject)
- **Automatic retry** with configurable limits
- **Complete audit trail** via `processing_log` with timestamps

### Key State Machine Patterns

1. **Enum lifecycles** — Define valid stages explicitly, not implicitly through agent routing
2. **Multi-branch conditional routing** — Real systems need 3+ branches, not just if/else
3. **Retry with back-off** — `retry_count` + `max_retries` prevents infinite retry loops
4. **Audit logging with `operator.add`** — Each node appends to the log; the reducer handles merging
5. **Error isolation** — Errors in one extraction path don't affect others
6. **Structured extraction** — Pydantic models ensure consistent output format per document type

### State Machine vs Agent Routing: When to Use Which

| Choose State Machine When... | Choose Agent Routing When... |
|----------------------------|---------------------------|
| Workflow has defined stages | Tasks are open-ended |
| You need audit compliance | Exploration is needed |
| Error handling must be explicit | LLM judgment drives flow |
| Transitions are rule-based | Routing requires reasoning |
| Debugging must be deterministic | Flexibility > predictability |

### Next Steps
- **Human in the Loop** → Add approval gates where humans review before the document advances
- **Combining Patterns** → Use a supervisor to manage the pre-processing, and a state machine for the post-processing pipeline